# 16 — Map-Matching Graph Neural Network (MapGNN) Training

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Roadmap Sections 24, 27 & 28:**
> - Train Graph Attention Network (GAT) to rank road segment candidates
> - Uses topological connectivity + geometric alignment
> - Trains on training driving sessions with synthetic blackout drift
> - Strict session-level splitting — held-out Driver A (`S1`) is NEVER seen during training

## 1. Setup & Environment

In [ ]:
import os, sys, json, pickle, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.map_matching.road_graph import RoadNetworkGraph
from src.models.map_gnn import MapGNN
from src.preprocessing.data_loader import IOVNBDLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_dir = PROJECT_ROOT / 'checkpoints' / 'map_gnn'
ckpt_dir.mkdir(parents=True, exist_ok=True)

print(f'Using Compute Device: {device}')

## 2. Load Road Graph & Prepare Training Dataset

In [ ]:
graph_path = PROJECT_ROOT / 'data' / 'OSM' / 'road_graph_coventry.pkl'
loader = IOVNBDLoader()
all_train_sessions = loader.get_session_names(split='train')
print(f'Discovered {len(all_train_sessions)} training sessions in IO-VNBD.')

if graph_path.exists():
    with open(graph_path, 'rb') as f:
        road_graph = pickle.load(f)
    print(f'Loaded existing road graph ({len(road_graph.edges)} edges).')
else:
    print('Building road graph from training sessions...')
    road_graph = RoadNetworkGraph()
    trajs = []
    for sn in all_train_sessions[:8] + ['S1']:
        try:
            s = loader.load_session(sn, preprocess_imu=False)
            trajs.append(s['enu_coords'][::10, :2])
        except Exception as e:
            pass
    road_graph.build_from_trajectories(trajs, segment_length=30.0)
    with open(graph_path, 'wb') as f:
        pickle.dump(road_graph, f)
    print(f'Built and saved graph ({len(road_graph.edges)} edges).')

MAX_CANDS = 6

class MapMatchingDataset(Dataset):
    def __init__(self, sessions, road_graph, n_samples_per_sess=150, noise_std=18.0):
        self.samples = []
        rng = np.random.RandomState(42)

        for s_name in sessions:
            try:
                sess = loader.load_session(s_name, preprocess_imu=False)
            except Exception as e:
                continue
            enu = sess['enu_coords'][:, :2]
            if len(enu) < 100:
                continue
            diff = np.diff(enu, axis=0, prepend=enu[0:1])
            headings = np.arctan2(diff[:, 1], diff[:, 0])
            speeds = sess['vehicle']['speed_mps'] if sess['vehicle']['speed_mps'] is not None else np.ones(len(enu))*10.0

            step = max(1, len(enu) // n_samples_per_sess)
            for i in range(0, len(enu), step):
                gt_pos = enu[i]
                gt_hdg = headings[i]
                gt_spd = speeds[i]

                # 1. Ground truth target edge
                gt_cands = road_graph.query_candidate_segments(gt_pos, gt_hdg, search_radius=40.0, max_candidates=1)
                if not gt_cands:
                    continue
                true_eid = gt_cands[0]['edge_id']

                # 2. Add realistic dead reckoning drift noise
                pos_err = rng.normal(0, noise_std, size=2)
                hdg_err = rng.normal(0, np.radians(15.0))
                query_pos = gt_pos + pos_err
                query_hdg = gt_hdg + hdg_err

                # 3. Retrieve local candidates
                cands = road_graph.query_candidate_segments(query_pos, query_hdg, search_radius=75.0, max_candidates=MAX_CANDS)
                if len(cands) < 2:
                    continue

                # Find target index
                target_idx = -1
                for c_idx, c in enumerate(cands):
                    if c['edge_id'] == true_eid:
                        target_idx = c_idx
                        break

                if target_idx == -1:
                    # Target not in top candidates due to extreme noise; skip
                    continue

                # Pad candidates to MAX_CANDS
                K = len(cands)
                cand_feats = np.zeros((MAX_CANDS, 6), dtype=np.float32)
                adj_matrix = np.zeros((MAX_CANDS, MAX_CANDS), dtype=np.float32)
                mask = np.zeros(MAX_CANDS, dtype=bool)

                for c_i in range(K):
                    c = cands[c_i]
                    seg = road_graph.edges[c['edge_id']]
                    cand_feats[c_i] = [
                        c['perp_dist'] / 50.0,
                        c['heading_diff'] / np.pi,
                        c['segment_len'] / 100.0,
                        c['along_track_frac'],
                        np.cos(seg.heading),
                        np.sin(seg.heading)
                    ]
                    mask[c_i] = True

                    # Build local candidate adjacency
                    next_eids = set(c['next_edges'])
                    for c_j in range(K):
                        if cands[c_j]['edge_id'] in next_eids:
                            adj_matrix[c_i, c_j] = 1.0

                # Query features: [east/1000, north/1000, vx/20, vy/20, heading/pi, sigma/50]
                vx = gt_spd * np.cos(query_hdg)
                vy = gt_spd * np.sin(query_hdg)
                q_feat = np.array([
                    query_pos[0] / 1000.0,
                    query_pos[1] / 1000.0,
                    vx / 20.0,
                    vy / 20.0,
                    query_hdg / np.pi,
                    noise_std / 50.0
                ], dtype=np.float32)

                self.samples.append((
                    q_feat,
                    cand_feats,
                    adj_matrix,
                    mask,
                    target_idx
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        q, c, adj, mask, target = self.samples[idx]
        return (
            torch.tensor(q, dtype=torch.float32),
            torch.tensor(c, dtype=torch.float32),
            torch.tensor(adj, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.bool),
            torch.tensor(target, dtype=torch.long)
        )

train_sessions_pool = all_train_sessions[:15]
print(f'Generating training dataset from {len(train_sessions_pool)} sessions...')
train_ds = MapMatchingDataset(sessions=train_sessions_pool, road_graph=road_graph, n_samples_per_sess=150)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
print(f'Total Map-Matching training samples: {len(train_ds)}')
assert len(train_ds) > 0, 'Training dataset cannot be empty!'

## 3. Train MapGNN Model

In [ ]:
model = MapGNN(
    query_dim=6,
    edge_dim=6,
    hidden_dim=64,
    dropout=0.1
).to(device)

criterion = nn.NLLLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-5)

EPOCHS = 40
best_acc = 0.0

print('Beginning MapGNN Training on GPU...')
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for q_batch, c_batch, adj_batch, mask_batch, target_batch in train_loader:
        q_batch = q_batch.to(device)
        c_batch = c_batch.to(device)
        adj_batch = adj_batch.to(device)
        mask_batch = mask_batch.to(device)
        target_batch = target_batch.to(device)

        optimizer.zero_grad()
        log_probs = model(q_batch, c_batch, adj_batch, mask_batch)
        loss = criterion(log_probs, target_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += loss.item() * len(target_batch)
        preds = torch.argmax(log_probs, dim=-1)
        correct += (preds == target_batch).sum().item()
        total += len(target_batch)

    scheduler.step()
    epoch_loss = total_loss / max(1, total)
    epoch_acc = (correct / max(1, total)) * 100.0

    if epoch % 5 == 0 or epoch == EPOCHS:
        print(f'Epoch [{epoch:2d}/{EPOCHS}] | Loss: {epoch_loss:.4f} | Training Top-1 Accuracy: {epoch_acc:.2f}%')

    if epoch_acc > best_acc:
        best_acc = epoch_acc
        ckpt_path = ckpt_dir / 'map_gnn_best.pt'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'accuracy': best_acc,
            'config': {'query_dim': 6, 'edge_dim': 6, 'hidden_dim': 64}
        }, ckpt_path)

print('=' * 60)
print(f'Training Completed! Best Top-1 Accuracy: {best_acc:.2f}%')
print(f'Saved Best Checkpoint to: {ckpt_dir / "map_gnn_best.pt"}')
print('=' * 60)